# 보호소 이미지 추출
이미지 가져온 후 크롭 + 리사이즈

# **->절대 모두 실행 금지<-**
608개 이미지가 10분동안 저장됨

pip 설치
- pip install ultralytics
- pip install python-dotenv

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
from ultralytics import YOLO

In [ ]:
model = YOLO("yolo11n.pt")

In [ ]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()  # 노트북 실행 위치에 .env 없으면 load_dotenv("ML/.env")

SERVICE_KEY = os.environ["SERVICE_KEY"]
BASE_URL = os.environ["BASE_URL"]
NUM_OF_ROWS = 1000

In [ ]:
# 1페이지만 테스트
def fetch_page(page_no: int, num_of_rows: int = NUM_OF_ROWS):

    url = (
        f"{BASE_URL}"
        f"?serviceKey={SERVICE_KEY}"
        f"&numOfRows={num_of_rows}"
        f"&pageNo={page_no}"
        f"&_type=json"
    )

    response = requests.get(
        url,
        timeout=20
    )

    print(response.status_code)

    response.raise_for_status()

    return response.json()

In [ ]:
data = fetch_page(1)

print(data.keys())
data

In [ ]:
# 사진이 3장 이상 있는 개체만 찾기
items = data["response"]["body"]["items"]["item"]

df = pd.DataFrame(items)

print(df.shape)
df.head()

In [ ]:
IMAGE_COLUMNS = [
    "popfile1",
    "popfile2",
    "popfile3",
    "popfile4",
    "popfile5",
    "popfile6",
    "popfile7",
    "popfile8",
]

In [ ]:
# 각 개체의 이미지 URL 개수 세기
def get_image_urls(row):
    urls = []

    for col in IMAGE_COLUMNS:
        if col not in row.index:
            continue

        url = row[col]

        if pd.notna(url):
            url = str(url).strip()

            if url:
                urls.append(url)
                
    # list(dict.fromkeys(urls)): 중복 키 제거
    return list(dict.fromkeys(urls))

In [ ]:
df["image_count"] = df.apply(
    lambda row: len(get_image_urls(row)),
    axis=1
)

df[
    ["desertionNo", "upKindNm", "image_count"]
].head()

In [ ]:
# 3장 이상인 데이터가 1개 이므로 그냥 모든 페이지로 확인할 예정
df["image_count"].value_counts().sort_index()

In [ ]:
# 전체 페이지 가져오기
all_items = []

page = 1

while True:
    data = fetch_page(page)

    body = data["response"]["body"]
    items = body["items"]["item"]

    if not items:
        break

    all_items.extend(items)

    print(f"{page}페이지 완료 / 누적 {len(all_items)}건")

    total_count = body["totalCount"]

    if len(all_items) >= total_count:
        break

    page += 1

In [ ]:
df = pd.DataFrame(all_items)

print("전체 개체 수:", len(df))

In [ ]:
df["image_count"] = df.apply(
    lambda row: len(get_image_urls(row)),
    axis=1
)

target_df = df[
    df["image_count"] >= 3
].copy()

print("사진 3장 이상 개체 수:", len(target_df))

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path("processed_animals")
OUTPUT_DIR.mkdir(exist_ok=True)

TARGET_SIZE = (224, 224)

In [ ]:
# URL에서 이미지 다운로드하는 함수
def download_image(url):
    try:
        response = requests.get(url, timeout=15)
        response.raise_for_status()

        image_array = np.frombuffer(
            response.content,
            dtype=np.uint8
        )

        image = cv2.imdecode(
            image_array,
            cv2.IMREAD_COLOR
        )

        return image

    except Exception as e:
        print("다운로드 실패:", url)
        print(e)

        return None

In [ ]:
# YOLO로 찾아서 Crop + Resize하는 함수
def crop_and_resize(image, animal_type):

    results = model(
        image,
        verbose=False
    )

    best_box = None
    best_conf = 0

    for result in results:

        for box in result.boxes:

            class_id = int(box.cls[0])
            class_name = model.names[class_id]

            # 보호소 데이터가 개라면 dog만
            if animal_type == "개":
                if class_name != "dog":
                    continue

            # 고양이라면 cat만
            elif animal_type == "고양이":
                if class_name != "cat":
                    continue

            else:
                continue

            confidence = float(box.conf[0])

            # 여러 마리가 탐지되면
            # 가장 신뢰도가 높은 객체 선택
            if confidence > best_conf:
                best_conf = confidence
                best_box = box

    # 탐지 실패
    if best_box is None:
        return None

    # YOLO Bounding Box
    x1, y1, x2, y2 = map(
        int,
        best_box.xyxy[0]
    )

    height, width = image.shape[:2]

    # 이미지 범위 밖으로 나가는 것을 방지
    x1 = max(0, x1)
    y1 = max(0, y1)
    x2 = min(width, x2)
    y2 = min(height, y2)

    # Crop
    cropped = image[
        y1:y2,
        x1:x2
    ]

    if cropped.size == 0:
        return None

    # 224 × 224 Resize
    resized = cv2.resize(
        cropped,
        TARGET_SIZE,
        interpolation=cv2.INTER_AREA
    )

    return resized

In [ ]:
row = target_df.iloc[0]

desertion_no = str(row["desertionNo"])
animal_type = row["upKindNm"]

image_urls = get_image_urls(row)

print("개체번호:", desertion_no)
print("동물종:", animal_type)
print("사진 수:", len(image_urls))

for url in image_urls:
    print(url)

In [ ]:
# 한 장 크롭 + 리사이즈 저장 (테스트)
test_dir = OUTPUT_DIR / desertion_no
test_dir.mkdir(parents=True, exist_ok=True)

success_count = 0

for i, url in enumerate(image_urls, start=1):

    print(f"{i}번째 이미지 처리 중...")

    # 이미지 다운로드
    image = download_image(url)

    if image is None:
        print(" → 다운로드 실패")
        continue

    # YOLO crop + resize
    processed = crop_and_resize(
        image,
        animal_type
    )

    if processed is None:
        print(" → YOLO 탐지 실패")
        continue

    # 저장
    save_path = test_dir / f"{i}.jpg"

    cv2.imwrite(
        str(save_path),
        processed
    )

    success_count += 1

    print(" → 저장 완료:", save_path)

print()
print("최종 성공:", success_count, "장")

In [ ]:
import matplotlib.pyplot as plt

for i, url in enumerate(image_urls, start=1):
    image = download_image(url)

    if image is None:
        continue

    processed = crop_and_resize(
        image,
        animal_type
    )

    if processed is None:
        image_rgb = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )

        plt.figure(figsize=(6, 6))
        plt.imshow(image_rgb)
        plt.title(f"{i}번 - YOLO 탐지 실패")
        plt.axis("off")
        plt.show()

In [ ]:
# 동물인건 인식하는데 개인줄을 모르고 있는중
for i, url in enumerate(image_urls, start=1):

    image = download_image(url)

    if image is None:
        continue

    results = model(
        image,
        verbose=False
    )

    print(f"\n{i}번째 이미지")

    for result in results:
        for box in result.boxes:

            class_id = int(box.cls[0])
            class_name = model.names[class_id]
            confidence = float(box.conf[0])

            print(
                class_name,
                round(confidence, 3)
            )

In [ ]:
# 리사이즈 비율 유지 + 회색 패딩
def resize_with_padding(image, target_size=(224, 224)):
    target_w, target_h = target_size
    h, w = image.shape[:2]

    scale = min(target_w / w, target_h / h)

    new_w = int(w * scale)
    new_h = int(h * scale)

    resized = cv2.resize(
        image,
        (new_w, new_h),
        interpolation=cv2.INTER_AREA
    )

    # 회색 패딩
    canvas = np.full(
        (target_h, target_w, 3),
        114,
        dtype=np.uint8
    )

    x_offset = (target_w - new_w) // 2
    y_offset = (target_h - new_h) // 2

    canvas[
        y_offset:y_offset + new_h,
        x_offset:x_offset + new_w
    ] = resized

    return canvas

In [ ]:
ANIMAL_CLASSES = {
    "bird",
    "cat",
    "dog",
    "horse",
    "sheep",
    "cow",
    "elephant",
    "bear",
    "zebra",
    "giraffe",
}


def crop_and_resize(image, animal_type):
    results = model(
        image,
        verbose=False
    )

    best_box = None
    largest_area = 0

    for result in results:
        for box in result.boxes:

            class_id = int(box.cls[0])
            class_name = model.names[class_id]

            if class_name not in ANIMAL_CLASSES:
                continue

            x1, y1, x2, y2 = map(
                int,
                box.xyxy[0]
            )

            area = (x2 - x1) * (y2 - y1)

            if area > largest_area:
                largest_area = area
                best_box = box

    if best_box is None:
        return None

    x1, y1, x2, y2 = map(
        int,
        best_box.xyxy[0]
    )

    height, width = image.shape[:2]

    x1 = max(0, x1)
    y1 = max(0, y1)
    x2 = min(width, x2)
    y2 = min(height, y2)

    cropped = image[y1:y2, x1:x2]

    if cropped.size == 0:
        return None

    # 비율 유지 + 회색 패딩
    resized = resize_with_padding(
        cropped,
        TARGET_SIZE
    )

    return resized

In [ ]:
# 한 장 크롭 + 리사이즈 저장 (테스트)
test_dir = OUTPUT_DIR / desertion_no
test_dir.mkdir(parents=True, exist_ok=True)

success_count = 0

for i, url in enumerate(image_urls, start=1):

    print(f"{i}번째 이미지 처리 중...")

    # 이미지 다운로드
    image = download_image(url)

    if image is None:
        print(" → 다운로드 실패")
        continue

    # YOLO crop + resize
    processed = crop_and_resize(
        image,
        animal_type
    )

    if processed is None:
        print(" → YOLO 탐지 실패")
        continue

    # 저장
    save_path = test_dir / f"{i}.jpg"

    cv2.imwrite(
        str(save_path),
        processed
    )

    success_count += 1

    print(" → 저장 완료:", save_path)

print()
print("최종 성공:", success_count, "장")

In [ ]:
saved_animals = []
failed_animals = []

for count, (_, row) in enumerate(
    target_df.iterrows(),
    start=1
):

    desertion_no = str(row["desertionNo"])
    animal_type = row["upKindNm"]

    image_urls = get_image_urls(row)

    print(
        f"[{count}/{len(target_df)}] "
        f"{desertion_no} / 원본 {len(image_urls)}장 처리 중..."
    )

    processed_images = []

    for i, url in enumerate(image_urls, start=1):

        image = download_image(url)

        if image is None:
            print(f"  {i}번 → 다운로드 실패")
            continue

        processed = crop_and_resize(
            image,
            animal_type
        )

        if processed is None:
            print(f"  {i}번 → 객체 탐지 실패")
            continue

        processed_images.append(processed)

    # 2장 이하이면 저장하지 않음
    if len(processed_images) <= 2:

        print(
            f"  → 제외: 성공 "
            f"{len(processed_images)}장\n"
        )

        failed_animals.append({
            "desertionNo": desertion_no,
            "original_count": len(image_urls),
            "processed_count": len(processed_images)
        })

        continue

    # 3장 이상일 때만 폴더 생성
    animal_dir = (
        OUTPUT_DIR
        / desertion_no
    )

    animal_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    for i, image in enumerate(
        processed_images,
        start=1
    ):

        save_path = (
            animal_dir
            / f"{i}.jpg"
        )

        cv2.imwrite(
            str(save_path),
            image
        )

    print(
        f"  → 저장 완료: "
        f"{len(processed_images)}장\n"
    )

    saved_animals.append({
        "desertionNo": desertion_no,
        "upKindNm": row["upKindNm"],
        "kindNm": row["kindNm"],
        "colorCd": row["colorCd"],
        "original_count": len(image_urls),
        "processed_count": len(processed_images)
    })

검증 필요하니 20개씩 이미지 띄우는 코드 생성

In [ ]:
from pathlib import Path
import cv2
import matplotlib.pyplot as plt

OUTPUT_DIR = Path("processed_animals")

In [ ]:
def review_animals(start=0, count=20):
    animal_dirs = sorted([
        p for p in OUTPUT_DIR.iterdir()
        if p.is_dir()
    ])

    end = min(start + count, len(animal_dirs))

    for idx in range(start, end):
        animal_dir = animal_dirs[idx]

        image_paths = sorted(
            animal_dir.glob("*.jpg")
        )

        if not image_paths:
            continue

        print(
            f"[{idx + 1}/{len(animal_dirs)}] "
            f"개체번호: {animal_dir.name}"
        )

        n = len(image_paths)

        fig, axes = plt.subplots(
            1,
            n,
            figsize=(4 * n, 4)
        )

        # 이미지가 1장일 때 처리
        if n == 1:
            axes = [axes]

        for ax, image_path in zip(
            axes,
            image_paths
        ):

            image = cv2.imread(
                str(image_path)
            )

            image = cv2.cvtColor(
                image,
                cv2.COLOR_BGR2RGB
            )

            ax.imshow(image)
            ax.set_title(
                image_path.name
            )
            ax.axis("off")

        plt.tight_layout()
        plt.show()

In [ ]:
review_animals(
    start=185,
    count=20
)

MegaDescriptor 테스트 결과(Top-1 0.72)가 배경 유사도 때문에 부풀려졌을 가능성이 있음.
→ 세그멘테이션으로 배경 제거 후 재측정하여 검증

In [ ]:
# 세그멘테이션 마스킹 함수
from pathlib import Path
import cv2
import numpy as np
from ultralytics import YOLO

seg_model = YOLO("yolo11n-seg.pt")   # 최초 실행 시 자동 다운로드

ANIMAL_CLASSES = {
    "bird", "cat", "dog", "horse", "sheep",
    "cow", "elephant", "bear", "zebra", "giraffe",
}
PAD_VALUE = 114   # 배경 채울 회색 (추출 파이프라인과 동일)


def mask_background(image, conf=0.15):
    """가장 큰 동물 인스턴스 마스크만 남기고 나머지는 회색."""
    results = seg_model(image, verbose=False, conf=conf)

    best_mask = None
    best_area = 0

    for result in results:
        if result.masks is None:
            continue

        for m, box in zip(result.masks.data, result.boxes):
            class_name = seg_model.names[int(box.cls[0])]
            if class_name not in ANIMAL_CLASSES:
                continue

            mask = m.cpu().numpy()
            area = float(mask.sum())

            if area > best_area:
                best_area = area
                best_mask = mask

    if best_mask is None:
        return None

    h, w = image.shape[:2]
    mask = cv2.resize(
        best_mask, (w, h),
        interpolation=cv2.INTER_NEAREST
    ).astype(bool)

    out = np.full_like(image, PAD_VALUE)
    out[mask] = image[mask]
    return out

In [ ]:
SRC_DIR = Path("processed_animals")
DST_DIR = Path("processed_animals_masked")
DST_DIR.mkdir(exist_ok=True)

n_ok, n_fail, failed = 0, 0, []

animal_dirs = sorted(p for p in SRC_DIR.iterdir() if p.is_dir())

for animal_dir in tqdm(animal_dirs, desc="배경 제거"):
    dst_animal = DST_DIR / animal_dir.name
    dst_animal.mkdir(exist_ok=True)

    for img_path in sorted(animal_dir.glob("*.jpg")):
        image = cv2.imread(str(img_path))
        masked = mask_background(image)

        if masked is None:
            n_fail += 1
            failed.append(str(img_path))
            continue

        cv2.imwrite(str(dst_animal / img_path.name), masked)
        n_ok += 1

# 마스킹 실패로 2장 이하 남은 개체 폴더 제거 (평가 왜곡 방지)
for dst_animal in list(DST_DIR.iterdir()):
    if dst_animal.is_dir() and len(list(dst_animal.glob("*.jpg"))) <= 2:
        for f in dst_animal.glob("*.jpg"):
            f.unlink()
        dst_animal.rmdir()

print(f"성공 {n_ok} / 실패 {n_fail}")
print("남은 개체 수:", sum(1 for p in DST_DIR.iterdir() if p.is_dir()))

In [ ]:
# 눈으로 확인
import matplotlib.pyplot as plt

sample_dirs = sorted(p for p in DST_DIR.iterdir() if p.is_dir())[:5]

for d in sample_dirs:
    paths = sorted(d.glob("*.jpg"))
    fig, axes = plt.subplots(2, len(paths), figsize=(4 * len(paths), 8))

    for k, p in enumerate(paths):
        orig = cv2.cvtColor(cv2.imread(str(SRC_DIR / d.name / p.name)), cv2.COLOR_BGR2RGB)
        mskd = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
        axes[0, k].imshow(orig); axes[0, k].axis("off"); axes[0, k].set_title(f"{d.name} 원본")
        axes[1, k].imshow(mskd); axes[1, k].axis("off"); axes[1, k].set_title("배경 제거")

    plt.tight_layout(); plt.show()